# Paper 2A — IJIES/INASS Major Revision Master Notebook — FINAL v1.1 (Round-3C extractor fix)

This notebook is a **single, reproducibility-oriented Colab workflow** built from the authoritative Paper 2 source notebooks supplied with the project:

- Round 2 v2.8: frozen Run82 method-parity implementation and definitive Run82 summaries.
- Round 3 v3.0: structured semantic/KG preservation evaluator.
- Round 3B v3.1: exact fair-budget Lead-W builder and paired comparison protocol.
- Round 3C v3.2.6: WebNLG gold-triple verification pipeline.

It executes the four remaining robustness tasks in sequence:

1. **Modern fair-budget baseline:** PRIMERA, evaluated using the same Round-3 structured-preservation evaluator.
2. **Alternative extractor robustness:** REBEL on an outcome-independent stratified test subset.
3. **WebNLG regrouping robustness:** repeated alternative bundle constructions, rerunning the original bundle → Run82/Lead-W → extraction → gold-evaluation pipeline.
4. **Mechanism ablation:** full Run82, no-topic, no-pattern, and no-redundancy variants using the original frozen summariser logic.

## Non-negotiable guards

- The historical Run82 seed is **42**.
- The original Run82 configuration remains **α=0.30, β=0.70, δ=1.00, 15 sentences, 25 topics**.
- No TEST-set result is used for parameter reselection.
- Seed **2026** is used only for new robustness-analysis sampling/resampling/regrouping, not as the historical Run82 seed.
- New results are reported whether they favor Run82 or not.
- The notebook never silently fabricates a result: failed provenance or parity checks stop the corresponding analysis.

## Recommended Colab runtime

Use a **GPU runtime**. PRIMERA, REBEL, WebNLG regrouping, and the ablations are computationally expensive. All long sections checkpoint to Drive/local output so interrupted runs can resume.

In [ ]:
#@title 0. Install dependencies
import os, sys, subprocess, warnings

os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning"
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "2")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

required = [
    "datasets>=3,<5",
    "spacy>=3.7,<4",
    "networkx>=3,<4",
    "scipy>=1.10,<2",
    "mlxtend>=0.23,<1",
    "transformers>=4.44,<5",
    "sentence-transformers>=3,<4",
    "accelerate>=0.34",
    "sentencepiece",
    "openpyxl",
    "pyarrow",
    "tqdm",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade-strategy", "only-if-needed", *required
])

model_check = subprocess.run(
    [sys.executable, "-c", "import en_core_web_sm"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if model_check.returncode != 0:
    subprocess.check_call([
        sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"
    ])

print("Dependencies ready.")

In [ ]:
#@title 1. Imports, paths, seeds, and experiment size
import os, re, json, math, time, gc, shutil, zipfile, hashlib, platform, urllib.request
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import spacy
import torch

from scipy.stats import wilcoxon, spearmanr
from datasets import load_dataset, Dataset
from tqdm.auto import tqdm
from IPython.display import display

SEED_ORIGINAL = 42       # historical Run82 / Round-3 statistical protocol
SEED_ROBUSTNESS = 2026     # only for new subset sampling / regrouping / robustness robustness
np.random.seed(SEED_ORIGINAL)

# Robustness-experiment defaults.
# 500 is intentionally much larger than the original 25-cluster BART sanity check,
# while remaining practical on Colab. Set 0 to run the entire 5,621-cluster test set.
MODERN_BASELINE_N = 500
REBEL_N = 500
ABLATION_N = 500

# True WebNLG regrouping repeats. 100 is the recommended final value.
WEBNLG_REGROUPINGS = 100

ROOT = Path("/content/Paper2A_IJIES_FINAL")
OUT = ROOT / "outputs"
CKPT = ROOT / "checkpoints"
OUT.mkdir(parents=True, exist_ok=True)
CKPT.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("CUDA:", torch.cuda.is_available())
print("Historical Run82 seed:", SEED_ORIGINAL)
print("Robustness-analysis seed:", SEED_ROBUSTNESS)

In [ ]:
#@title 2. Upload the four definitive result packages
# Required:
# - Paper2_Round2_DEFINITIVE_Run82_METHOD_PARITY_v28.zip
# - Paper2_Round3_Semantic_KG_Preservation.zip
# - Paper2_Round3B_Fair_Budget_LeadW.zip
# - Paper2_Round3C_WebNLG_Gold_Triple_Verification.zip

from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded))

In [ ]:
#@title 3. Locate, extract, and verify definitive packages
def locate_zip(fragment):
    hits = [
        Path("/content") / n for n in os.listdir("/content")
        if fragment.lower() in n.lower() and n.lower().endswith(".zip")
    ]
    if not hits:
        raise FileNotFoundError(fragment)
    return hits[0]

Z_R2 = locate_zip("Round2_DEFINITIVE_Run82_METHOD_PARITY_v28")
Z_R3 = locate_zip("Round3_Semantic_KG_Preservation")
Z_R3B = locate_zip("Round3B_Fair_Budget_LeadW")
Z_R3C = locate_zip("Round3C_WebNLG_Gold_Triple_Verification")

def unzip(zp, name):
    d = ROOT / name
    if d.exists():
        shutil.rmtree(d)
    d.mkdir()
    with zipfile.ZipFile(zp) as z:
        z.extractall(d)
    return d

D_R2 = unzip(Z_R2, "round2")
D_R3 = unzip(Z_R3, "round3")
D_R3B = unzip(Z_R3B, "round3b")
D_R3C = unzip(Z_R3C, "round3c")

def one(folder, pattern):
    hits = list(folder.rglob(pattern))
    if len(hits) != 1:
        raise RuntimeError(f"{pattern}: expected one file, found {len(hits)}")
    return hits[0]

S2 = json.loads(one(D_R2, "ROUND2_DEFINITIVE_STATUS.json").read_text())
S3 = json.loads(one(D_R3, "ROUND3_FINALIZATION_STATUS.json").read_text())
S3B = json.loads(one(D_R3B, "ROUND3B_FINALIZATION_STATUS.json").read_text())
S3C = json.loads(one(D_R3C, "ROUND3C_FINALIZATION_STATUS.json").read_text())

guard = pd.DataFrame([
    ["Round2 definitive", S2.get("round2_definitive_complete") is True],
    ["Round3 finalized", S3.get("round3_finalized") is True],
    ["Round3B finalized", S3B.get("round3b_finalized") is True],
    ["Round3C finalized", S3C.get("round3c_finalized") is True],
    ["Round2 frozen run 82", int(S2.get("selected_validation_run_order", -1)) == 82],
    ["Round3 frozen run 82", int(S3.get("frozen_run_order", -1)) == 82],
    ["Round3B frozen run 82", int(S3B.get("frozen_run_order", -1)) == 82],
    ["Round3C frozen run 82", int(S3C.get("frozen_run_order", -1)) == 82],
    ["Round2 no retuning", S2.get("parameter_retuning_performed") is False],
    ["Round3 no retuning", S3.get("parameter_retuning_performed") is False],
    ["Round3B no retuning", S3B.get("parameter_retuning_performed") is False],
    ["Round3C no retuning", S3C.get("parameter_retuning_performed") is False],
], columns=["guard", "pass"])

display(guard)
if not guard["pass"].all():
    raise RuntimeError("Definitive provenance guard failed.")

guard.to_csv(OUT / "00_definitive_provenance_guard.csv", index=False)
print("PASS: all definitive provenance guards.")

In [ ]:
#@title 4. Load definitive evidence and resolve 0.3997 vs 0.3994
R2 = pd.read_csv(one(D_R2, "round2_test_cluster_details.csv"))
R2_AGG = pd.read_csv(one(D_R2, "paper2_definitive_test_result.csv"))
R3 = pd.read_csv(one(D_R3, "round3_cluster_metrics_all.csv"))
R3B = pd.read_csv(one(D_R3B, "round3b_leadw_cluster_metrics_all.csv"))
R3B_CMP = pd.read_csv(one(D_R3B, "round3b_run82_vs_leadw_primary_comparison.csv"))
R3C = pd.read_csv(one(D_R3C, "round3c_bundle_results.csv"))
R3C_CMP = pd.read_csv(one(D_R3C, "round3c_run82_vs_leadw_gold_comparison.csv"))

assert len(R2) == len(R3) == len(R3B) == 5621
assert len(R3C) == 71

round2_pres = float(R2_AGG.iloc[0]["preservation_f1"])
round3_exact = float(R3["run82_exact_triple_f1"].mean())

metric_reconciliation = pd.DataFrame([
    {
        "metric_object": "Round2 preservation_f1",
        "value": round2_pres,
        "rounded_4dp": round(round2_pres, 4),
        "definition": "Round-2 method-parity dependency-triple aggregate preservation F1",
    },
    {
        "metric_object": "Round3 exact_triple_f1",
        "value": round3_exact,
        "rounded_4dp": round(round3_exact, 4),
        "definition": "Round-3 explicit exact structured-triple preservation F1",
    },
])
display(metric_reconciliation)
metric_reconciliation.to_csv(OUT / "01_metric_reconciliation_03997_03994.csv", index=False)

print("For the Round-3 fair-budget table, use exact-triple F1 =", f"{round3_exact:.4f}")

## Robustness 1 — Comment 4 wording

**Recommended response**

> We traced the apparent discrepancy to two distinct metric objects rather than to rounding of one statistic. The definitive Round-2 v2.8 method-parity artifact reports the earlier aggregate dependency-triple preservation F1, whereas Round 3 decomposes structured preservation and reports the explicitly defined exact-triple F1. The revised fair-budget analysis uses the Round-3 definition consistently. We therefore report the Run82 exact-triple F1 as 0.3994 in the Abstract, Results, and fair-budget tables, while explicitly distinguishing the earlier Round-2 preservation metric where relevant.

In [ ]:
#@title 5. Reload the exact Multi-News test source and build an auditable text cache
DATASET_REPO = S2.get("dataset_repository", "Awesome075/multi_news_parquet")
raw_test = load_dataset(DATASET_REPO, split="test")

def clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\u00a0", " ")).strip()

source_rows = []
for idx, row in enumerate(raw_test):
    raw_document = clean_text(row.get("document", ""))
    reference = clean_text(row.get("summary", ""))
    documents = [
        clean_text(x)
        for x in raw_document.split("|||||")
        if clean_text(x)
    ]
    if not documents or not reference:
        continue
    source_rows.append({
        "cluster_id": f"test_{idx:05d}",
        "raw_index": idx,
        "documents": documents,
        "source_text": clean_text(" ".join(documents)),
        "reference_summary": reference,
    })

SOURCE = pd.DataFrame(source_rows)
assert len(SOURCE) == 5621

CACHE = (
    R2[["cluster_id","raw_index","predicted_summary","original_words","summary_words"]]
    .rename(columns={"predicted_summary":"run82_text","summary_words":"run82_words"})
    .merge(SOURCE, on=["cluster_id","raw_index"], how="inner", validate="one_to_one")
    .merge(
        R3B[["cluster_id","leadw_words","leadw_sentences","absolute_budget_mismatch_pct"]],
        on="cluster_id", how="inner", validate="one_to_one"
    )
)

assert len(CACHE) == 5621
assert (CACHE["run82_text"].str.split().str.len() == CACHE["run82_words"]).all()

CACHE["compression_gain"] = 1 - CACHE["run82_words"] / CACHE["original_words"]
CACHE.to_pickle(OUT / "paper2a_text_cache_base.pkl")
print("PASS: source documents and exact Run82 summaries recovered for 5,621 clusters.")

# Authoritative frozen Run82 implementation

The following code is embedded from the supplied Round-2 v2.8 method-parity notebook.  
No parameters are retuned. The only additional function later in this notebook is a parameterized wrapper for mechanism ablation.

In [ ]:
#@title 6. Historical frozen constants used by the embedded Round-2 implementation
ALPHA = 0.30
BETA = 0.70
DELTA = 1.00
SUMMARY_BUDGET = 15
N_TOPICS = 25
SEED = 42

In [ ]:
#@title 4. Embedded Round-1 v2.3/v2.4 method-parity summariser
import signal
from collections import Counter
import spacy

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth

# Exact experimental constants used by the finalized 120-scenario Round 1.
MIN_PATTERN_SUPPORT = 0.08
MIN_PATTERN_OCCURRENCES = 2
MAX_PATTERN_LENGTH = 3
MAX_TRANSACTION_ITEMS = 20
MAX_FREQUENT_ITEMSETS = 500_000
FP_TIMEOUT_SECONDS = 300
CANDIDATE_POOL_PER_TOPIC = 50
MAX_SENTENCES_FOR_SIMILARITY = 220
LDA_MAX_FEATURES = 5000
LDA_MAX_ITER = 10
LDA_N_JOBS = 1

# Parser active for sentence boundaries and dependency triples.
nlp_dep = spacy.load("en_core_web_sm", disable=["ner"])

# Round-1 lexical transaction pipeline: parser/NER disabled, tagger+lemmatizer retained.
nlp_lex = spacy.load("en_core_web_sm", disable=["parser", "ner"])
if "sentencizer" not in nlp_lex.pipe_names:
    nlp_lex.add_pipe("sentencizer")

def safe_sentencize(documents):
    if isinstance(documents, str):
        documents = [documents]
    sentences = []
    for document in documents:
        text = clean_text(document)
        if not text:
            continue
        doc = nlp_dep(text)
        for sent in doc.sents:
            s = clean_text(sent.text)
            if s:
                sentences.append(s)
    return sentences

def _normalise_rows(matrix):
    arr = np.asarray(matrix, dtype=float)
    denom = arr.sum(axis=1, keepdims=True)
    denom[denom == 0] = 1.0
    return arr / denom

def fit_topic_model(sentences, n_topics=N_TOPICS, seed=SEED):
    n = len(sentences)
    if n == 0:
        return {
            "topic_score": np.array([], dtype=float),
            "theta": np.empty((0, 1), dtype=float),
            "phi": np.ones((1, 1), dtype=float),
            "vectorizer": None,
            "X": None,
            "feature_names": np.array(["fallback"], dtype=str),
            "dominant_topic": np.array([], dtype=int),
        }

    # Round-1 vectorizer contract.
    vectorizer = CountVectorizer(
        stop_words="english",
        lowercase=True,
        max_features=LDA_MAX_FEATURES,
        ngram_range=(1, 1),
        min_df=1,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    )

    try:
        X = vectorizer.fit_transform(sentences)
    except ValueError:
        X = None

    if X is None or X.shape[1] == 0:
        return {
            "topic_score": np.ones(n, dtype=float),
            "theta": np.ones((n, 1), dtype=float),
            "phi": np.ones((1, 1), dtype=float),
            "vectorizer": None,
            "X": None,
            "feature_names": np.array(["fallback"], dtype=str),
            "dominant_topic": np.zeros(n, dtype=int),
        }

    effective_topics = max(
        1,
        min(int(n_topics), int(X.shape[0]), max(1, int(X.shape[1]))),
    )

    lda = LatentDirichletAllocation(
        n_components=effective_topics,
        random_state=int(seed),
        learning_method="batch",
        max_iter=LDA_MAX_ITER,
        n_jobs=LDA_N_JOBS,
    )
    theta = lda.fit_transform(X)
    phi = _normalise_rows(lda.components_)
    feature_names = vectorizer.get_feature_names_out()

    # Topic significance = mean strongest topic-word salience across unique words.
    X_bin = (X > 0).astype(float)
    word_topic_salience = phi.max(axis=0)
    raw_ts = np.asarray(X_bin @ word_topic_salience).ravel()
    token_counts = np.asarray(X_bin.sum(axis=1)).ravel()
    token_counts[token_counts == 0] = 1.0
    topic_score = raw_ts / token_counts
    dominant_topic = theta.argmax(axis=1)

    return {
        "topic_score": np.asarray(topic_score, dtype=float),
        "theta": np.asarray(theta, dtype=float),
        "phi": np.asarray(phi, dtype=float),
        "vectorizer": vectorizer,
        "X": X,
        "feature_names": np.asarray(feature_names),
        "dominant_topic": np.asarray(dominant_topic, dtype=int),
    }

def _topic_word_weight_map(topic_model):
    feature_names = topic_model.get("feature_names")
    phi = topic_model.get("phi")
    if feature_names is None or phi is None or len(feature_names) == 0:
        return {}
    strongest = np.asarray(phi).max(axis=0)
    return {
        str(term): float(weight)
        for term, weight in zip(feature_names, strongest)
    }

def lexical_transactions(sentences, batch_size=128):
    """Round-1 lemmatised lexical transactions."""
    transactions = []
    docs = nlp_lex.pipe(
        (str(sentence).lower() for sentence in sentences),
        batch_size=int(batch_size),
    )
    for doc in docs:
        tokens = []
        for token_obj in doc:
            token = (token_obj.lemma_ or token_obj.text).lower().strip()
            if (
                token_obj.is_alpha
                and not token_obj.is_stop
                and len(token) > 2
            ):
                tokens.append(token)
        transactions.append(sorted(set(tokens)))
    return transactions

class _FPGrowthTimeout(Exception):
    pass

def _fp_timeout_handler(signum, frame):
    raise _FPGrowthTimeout("FP-Growth exceeded timeout")

def pattern_scores(sentences, topic_model, min_support=MIN_PATTERN_SUPPORT):
    if not sentences:
        return np.array([], dtype=float), []

    raw_transactions = lexical_transactions(sentences)
    if not any(raw_transactions):
        return np.zeros(len(sentences), dtype=float), []

    topic_weight = _topic_word_weight_map(topic_model)

    # Round-1 deterministic topic-salience transaction pruning.
    transactions = []
    for items in raw_transactions:
        items = list(items)
        if len(items) > MAX_TRANSACTION_ITEMS:
            items = sorted(
                items,
                key=lambda term: (-topic_weight.get(term, 0.0), term),
            )[:MAX_TRANSACTION_ITEMS]
        transactions.append(sorted(set(items)))

    encoder = TransactionEncoder()
    encoded = encoder.fit(transactions).transform(transactions)
    frame = pd.DataFrame(encoded, columns=encoder.columns_)

    required_occurrences = min(
        int(MIN_PATTERN_OCCURRENCES),
        max(1, len(transactions)),
    )
    effective_support = max(
        float(min_support),
        float(required_occurrences) / max(1, len(transactions)),
    )

    previous_handler = signal.signal(signal.SIGALRM, _fp_timeout_handler)
    signal.alarm(int(FP_TIMEOUT_SECONDS))
    try:
        try:
            freq = fpgrowth(
                frame,
                min_support=effective_support,
                use_colnames=True,
                max_len=int(MAX_PATTERN_LENGTH),
            )
        except _FPGrowthTimeout as exc:
            raise RuntimeError(
                "FP-Growth timeout. No parameter was changed; "
                "the definitive run stopped explicitly."
            ) from exc
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, previous_handler)

    if freq.empty:
        return np.zeros(len(sentences), dtype=float), []

    # Round 1 stopped explicitly rather than truncating pathological output.
    if len(freq) > MAX_FREQUENT_ITEMSETS:
        raise RuntimeError(
            f"FP-Growth returned {len(freq):,} frequent itemsets, exceeding "
            f"the explicit cap {MAX_FREQUENT_ITEMSETS:,}. "
            "No parameter was changed."
        )

    column_index = {
        str(term): idx for idx, term in enumerate(encoder.columns_)
    }
    encoded_bool = np.asarray(encoded, dtype=bool)

    pattern_records = []
    scores = np.zeros(len(sentences), dtype=float)

    for row in freq.itertuples(index=False):
        itemset = frozenset(str(x) for x in row.itemsets)
        if not itemset:
            continue

        coherence = float(np.mean([
            topic_weight.get(term, 0.0) for term in itemset
        ]))
        support = float(row.support)
        weight = support * coherence

        pattern_records.append({
            "items": itemset,
            "support": support,
            "topic_coherence": coherence,
            "weight": weight,
        })

        columns = [
            column_index[term]
            for term in itemset
            if term in column_index
        ]
        if len(columns) != len(itemset):
            continue

        active = encoded_bool[:, columns].all(axis=1)
        scores[active] += weight

    return scores, pattern_records

def build_candidate_pool(topic_model, per_topic=CANDIDATE_POOL_PER_TOPIC):
    theta = np.asarray(topic_model.get("theta"))
    if theta.size == 0:
        return np.array([], dtype=int)

    selected = set()
    per_topic = max(1, int(per_topic))
    for topic_id in range(theta.shape[1]):
        order = np.argsort(-theta[:, topic_id])
        for idx in order[:per_topic]:
            selected.add(int(idx))
    return np.array(sorted(selected), dtype=int)

def _paper1_similarity(sentences, candidate_indices=None):
    """Round-1 candidate-only TF-IDF cosine matrix."""
    if candidate_indices is None:
        candidate_indices = list(range(len(sentences)))
    candidate_indices = list(candidate_indices)

    if not candidate_indices:
        return np.empty((0, 0)), candidate_indices

    if len(candidate_indices) > MAX_SENTENCES_FOR_SIMILARITY:
        candidate_indices = candidate_indices[:MAX_SENTENCES_FOR_SIMILARITY]

    subset = [sentences[i] for i in candidate_indices]

    if len(subset) == 1:
        return np.ones((1, 1), dtype=np.float32), candidate_indices

    try:
        X = TfidfVectorizer(
            stop_words="english",
            lowercase=True,
            token_pattern=r"(?u)\b\w\w+\b",
        ).fit_transform(subset)
        sim = cosine_similarity(
            X,
            dense_output=True,
        ).astype(np.float32, copy=False)
        return sim, candidate_indices
    except ValueError:
        return np.eye(len(subset), dtype=np.float32), candidate_indices

def generate_summary(documents):
    """
    Frozen Run-82 summariser with Round-1 method parity.

    Score(s) = alpha*TS(s) + beta*PR(s) - delta*Red(s,S)
    """
    sentences = safe_sentencize(documents)
    if not sentences:
        return "", {
            "n_sentences": 0,
            "patterns_found": 0,
            "candidate_count": 0,
        }

    target = max(1, min(int(SUMMARY_BUDGET), len(sentences)))

    # For short clusters the extractive summary is all available sentences.
    if len(sentences) <= target:
        return " ".join(sentences), {
            "n_sentences": int(len(sentences)),
            "patterns_found": 0,
            "candidate_count": int(len(sentences)),
        }

    topic_model = fit_topic_model(sentences, n_topics=N_TOPICS, seed=SEED)
    topic = np.asarray(topic_model["topic_score"], dtype=np.float32)
    patt, patterns = pattern_scores(
        sentences,
        topic_model=topic_model,
        min_support=MIN_PATTERN_SUPPORT,
    )
    patt = np.asarray(patt, dtype=np.float32)

    candidates = build_candidate_pool(
        topic_model,
        per_topic=CANDIDATE_POOL_PER_TOPIC,
    )
    if candidates.size == 0 or len(candidates) < target:
        candidates = np.arange(len(sentences), dtype=int)

    # CRITICAL Round-1 parity: rank by relevance BEFORE the 220-sentence cap.
    base = (
        ALPHA * topic
        + BETA * patt
    )
    ordered_candidates = sorted(
        (int(i) for i in candidates),
        key=lambda i: (-float(base[i]), int(i)),
    )

    sim, active_candidates = _paper1_similarity(
        sentences,
        ordered_candidates,
    )
    active_candidates = list(active_candidates)
    position = {
        idx: pos for pos, idx in enumerate(active_candidates)
    }

    selected = []
    remaining = set(active_candidates)

    while remaining and len(selected) < target:
        best_idx = None
        best_score = -np.inf

        for idx in sorted(remaining):
            if selected:
                i = position[idx]
                redundancy = max(
                    float(sim[i, position[j]])
                    for j in selected
                )
            else:
                redundancy = 0.0

            score = float(
                ALPHA * topic[idx]
                + BETA * patt[idx]
                - DELTA * redundancy
            )

            if (
                score > best_score
                or (
                    np.isclose(score, best_score)
                    and (best_idx is None or idx < best_idx)
                )
            ):
                best_score = score
                best_idx = int(idx)

        if best_idx is None:
            break

        selected.append(best_idx)
        remaining.remove(best_idx)

    # Extractive presentation follows original document order.
    summary = " ".join(
        sentences[i]
        for i in sorted(selected)
    )

    return summary, {
        "n_sentences": int(len(sentences)),
        "patterns_found": int(len(patterns)),
        "candidate_count": int(len(active_candidates)),
    }

print("Round-1 method-parity Run-82 summariser ready.")

# Authoritative Round-3 semantic/KG preservation evaluator

The next two cells are embedded from the supplied Round-3 v3.0 notebook and are used unchanged for the main structured-preservation metrics.

In [ ]:
#@title 8. Imports required by the authoritative Round-3 evaluator
from collections import Counter, defaultdict
import networkx as nx
import spacy

In [ ]:
#@title 4. Deterministic semantic/KG extractor and preservation metrics
nlp = spacy.load("en_core_web_sm")
nlp.max_length = 5_000_000

# ---------------------------
# Canonicalisation
# ---------------------------

COARSE_ENTITY_MAP = {
    "PERSON": "PERSON",
    "NORP": "GROUP",
    "FAC": "LOCATION",
    "ORG": "ORG",
    "GPE": "LOCATION",
    "LOC": "LOCATION",
    "PRODUCT": "PRODUCT",
    "EVENT": "EVENT",
    "WORK_OF_ART": "ARTIFACT",
    "LAW": "ARTIFACT",
    "LANGUAGE": "LANGUAGE",
    "DATE": "TIME",
    "TIME": "TIME",
    "PERCENT": "QUANTITY",
    "MONEY": "QUANTITY",
    "QUANTITY": "QUANTITY",
    "ORDINAL": "QUANTITY",
    "CARDINAL": "QUANTITY",
}

def norm_space(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def canonical_text(text):
    text = norm_space(text).lower()
    text = re.sub(r"[^a-z0-9\s_-]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def content_lemma(tokens):
    vals = []
    for tok in tokens:
        if tok.is_space or tok.is_punct:
            continue
        if tok.is_stop and tok.pos_ not in {"PROPN", "NOUN", "NUM"}:
            continue
        lemma = canonical_text(tok.lemma_ or tok.text)
        if lemma:
            vals.extend(lemma.split())
    return " ".join(vals).strip()

def span_semantic_type(span):
    labels = [
        COARSE_ENTITY_MAP.get(ent.label_, "OTHER_ENTITY")
        for ent in span.ents
    ]
    if labels:
        return Counter(labels).most_common(1)[0][0]

    root = span.root
    if root.pos_ == "PROPN":
        return "ENTITY"
    if root.pos_ in {"NOUN", "PRON"}:
        return "CONCEPT"
    if root.pos_ == "NUM":
        return "QUANTITY"
    return "OTHER"

# ---------------------------
# Concept & entity extraction
# ---------------------------

def extract_entities(doc):
    entities = {}
    for ent in doc.ents:
        canon = content_lemma(ent)
        if not canon:
            canon = canonical_text(ent.text)
        if not canon:
            continue
        entities[canon] = {
            "label": canon,
            "surface": norm_space(ent.text),
            "type": COARSE_ENTITY_MAP.get(ent.label_, "OTHER_ENTITY"),
        }
    return entities

def extract_concepts(doc):
    entity_token_ids = set()
    for ent in doc.ents:
        entity_token_ids.update(range(ent.start, ent.end))

    concepts = {}
    for chunk in doc.noun_chunks:
        token_ids = set(range(chunk.start, chunk.end))
        # Keep concepts separate from chunks that are fully a named entity.
        if token_ids and token_ids.issubset(entity_token_ids):
            continue

        canon = content_lemma(chunk)
        if not canon:
            continue

        # Avoid one-character/no-content artefacts.
        if len(canon) < 2:
            continue

        concepts[canon] = {
            "label": canon,
            "surface": norm_space(chunk.text),
            "type": "CONCEPT",
        }
    return concepts

# ---------------------------
# Dependency triples
# ---------------------------

def subtree_span(token):
    toks = sorted(list(token.subtree), key=lambda x: x.i)
    if not toks:
        return token.doc[token.i:token.i+1]
    return token.doc[toks[0].i:toks[-1].i + 1]

def canonical_relation(root, prep=None):
    base = canonical_text(root.lemma_ or root.text)
    if prep is not None:
        p = canonical_text(prep.lemma_ or prep.text)
        return f"{base}_{p}" if p else base
    return base

def extract_triples(doc):
    triples = []
    seen = set()

    for sent in doc.sents:
        roots = [
            token for token in sent
            if token.dep_ == "ROOT" and token.pos_ in {"VERB", "AUX"}
        ]

        for root in roots:
            children = list(root.children)

            subjects = [
                t for t in children
                if t.dep_ in {"nsubj", "nsubjpass", "csubj"}
            ]
            objects = [
                t for t in children
                if t.dep_ in {"dobj", "obj", "attr", "oprd", "dative"}
            ]

            for subj in subjects:
                sspan = subtree_span(subj)
                s = content_lemma(sspan) or canonical_text(sspan.text)
                stype = span_semantic_type(sspan)

                for obj in objects:
                    ospan = subtree_span(obj)
                    o = content_lemma(ospan) or canonical_text(ospan.text)
                    otype = span_semantic_type(ospan)
                    r = canonical_relation(root)

                    key = (s, r, o)
                    if all(key) and key not in seen:
                        seen.add(key)
                        triples.append({
                            "s": s, "r": r, "o": o,
                            "stype": stype, "otype": otype,
                        })

                for prep in [t for t in children if t.dep_ == "prep"]:
                    prep_objects = [
                        t for t in prep.children
                        if t.dep_ in {"pobj", "obj"}
                    ]
                    for obj in prep_objects:
                        ospan = subtree_span(obj)
                        o = content_lemma(ospan) or canonical_text(ospan.text)
                        otype = span_semantic_type(ospan)
                        r = canonical_relation(root, prep)

                        key = (s, r, o)
                        if all(key) and key not in seen:
                            seen.add(key)
                            triples.append({
                                "s": s, "r": r, "o": o,
                                "stype": stype, "otype": otype,
                            })

    return triples

# ---------------------------
# Generic set preservation
# ---------------------------

def prf_set(candidate, reference):
    cand = set(candidate)
    ref = set(reference)
    tp = len(cand & ref)
    fp = len(cand - ref)
    fn = len(ref - cand)
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    j = len(cand & ref) / len(cand | ref) if (cand | ref) else 1.0
    return {
        "precision": float(p),
        "recall": float(r),
        "f1": float(f),
        "jaccard": float(j),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
    }

def exact_triple_keys(triples):
    return {(t["s"], t["r"], t["o"]) for t in triples}

def relation_keys(triples):
    return {t["r"] for t in triples}

# ---------------------------
# Soft lexical-semantic triple matching
# ---------------------------

def token_set(text):
    return set(canonical_text(text).split())

def jaccard_text(a, b):
    A, B = token_set(a), token_set(b)
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def relation_base(rel):
    return canonical_text(rel).split("_", 1)[0]

def relation_similarity(a, b):
    if a == b:
        return 1.0
    if relation_base(a) == relation_base(b) and relation_base(a):
        return 0.85
    return 0.0

def soft_triple_prf(candidate_triples, reference_triples):
    # Deterministic one-to-one greedy matching, candidate against source reference.
    pairs = []
    for i, c in enumerate(candidate_triples):
        for j, r in enumerate(reference_triples):
            rs = relation_similarity(c["r"], r["r"])
            if rs < 0.85:
                continue
            ss = jaccard_text(c["s"], r["s"])
            os_ = jaccard_text(c["o"], r["o"])
            if ss < 0.50 or os_ < 0.50:
                continue
            score = (ss + rs + os_) / 3.0
            if score >= 0.70:
                pairs.append((score, i, j))

    pairs.sort(key=lambda x: (-x[0], x[1], x[2]))
    used_c, used_r = set(), set()
    matches = 0

    for score, i, j in pairs:
        if i in used_c or j in used_r:
            continue
        used_c.add(i)
        used_r.add(j)
        matches += 1

    p = matches / len(candidate_triples) if candidate_triples else 0.0
    r = matches / len(reference_triples) if reference_triples else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0

    return {
        "precision": float(p),
        "recall": float(r),
        "f1": float(f),
        "matches": int(matches),
    }

# ---------------------------
# Domain-range consistency
# ---------------------------

def domain_range_consistency(candidate_triples, reference_triples):
    by_rel_subject_types = defaultdict(set)
    by_rel_object_types = defaultdict(set)
    by_rel_pairs = defaultdict(set)

    for t in reference_triples:
        by_rel_subject_types[t["r"]].add(t["stype"])
        by_rel_object_types[t["r"]].add(t["otype"])
        by_rel_pairs[t["r"]].add((t["stype"], t["otype"]))

    if not candidate_triples:
        return {
            "domain_accuracy": 0.0,
            "range_accuracy": 0.0,
            "pair_accuracy": 0.0,
            "violation_rate": 0.0,
            "evaluated_relations": 0,
        }

    domain_ok = 0
    range_ok = 0
    pair_ok = 0

    for t in candidate_triples:
        rel = t["r"]
        if t["stype"] in by_rel_subject_types.get(rel, set()):
            domain_ok += 1
        if t["otype"] in by_rel_object_types.get(rel, set()):
            range_ok += 1
        if (t["stype"], t["otype"]) in by_rel_pairs.get(rel, set()):
            pair_ok += 1

    n = len(candidate_triples)
    pair_acc = pair_ok / n

    return {
        "domain_accuracy": domain_ok / n,
        "range_accuracy": range_ok / n,
        "pair_accuracy": pair_acc,
        "violation_rate": 1.0 - pair_acc,
        "evaluated_relations": n,
    }

# ---------------------------
# Graph structure
# ---------------------------

def graph_from_triples(triples):
    G = nx.Graph()
    for t in triples:
        s, o = t["s"], t["o"]
        if s:
            G.add_node(s)
        if o:
            G.add_node(o)
        if s and o:
            G.add_edge(s, o, relation=t["r"])
    return G

def largest_component_size(G):
    if G.number_of_nodes() == 0:
        return 0
    return max(len(c) for c in nx.connected_components(G))

def structural_metrics(candidate_triples, reference_triples, top_k=10):
    Gc = graph_from_triples(candidate_triples)
    Gr = graph_from_triples(reference_triples)

    nodes_c = set(Gc.nodes())
    nodes_r = set(Gr.nodes())
    edges_c = exact_triple_keys(candidate_triples)
    edges_r = exact_triple_keys(reference_triples)

    node_retention = len(nodes_c & nodes_r) / len(nodes_r) if nodes_r else 1.0
    edge_retention = len(edges_c & edges_r) / len(edges_r) if edges_r else 1.0
    edge_jaccard = len(edges_c & edges_r) / len(edges_c | edges_r) if (edges_c | edges_r) else 1.0

    rel_c = relation_keys(candidate_triples)
    rel_r = relation_keys(reference_triples)
    rel_retention = len(rel_c & rel_r) / len(rel_r) if rel_r else 1.0

    lcc_r = largest_component_size(Gr)
    lcc_c = largest_component_size(Gc)
    lcc_ratio = lcc_c / lcc_r if lcc_r else 1.0

    if Gr.number_of_nodes() > 0:
        ranked_r = [
            n for n, d in sorted(
                Gr.degree,
                key=lambda x: (-x[1], x[0])
            )[:top_k]
        ]
    else:
        ranked_r = []

    if Gc.number_of_nodes() > 0:
        ranked_c = [
            n for n, d in sorted(
                Gc.degree,
                key=lambda x: (-x[1], x[0])
            )[:top_k]
        ]
    else:
        ranked_c = []

    denom = min(top_k, len(ranked_r))
    central_at_k = (
        len(set(ranked_r) & set(ranked_c)) / denom if denom else 1.0
    )

    return {
        "node_retention": float(node_retention),
        "edge_retention": float(edge_retention),
        "edge_jaccard": float(edge_jaccard),
        "relation_type_retention": float(rel_retention),
        "lcc_ratio": float(lcc_ratio),
        "central_node_preservation_at10": float(central_at_k),
        "original_nodes": int(Gr.number_of_nodes()),
        "candidate_nodes": int(Gc.number_of_nodes()),
        "original_edges": int(len(edges_r)),
        "candidate_edges": int(len(edges_c)),
    }

print("Semantic/KG extractor ready.")

In [ ]:
#@title 5. Lead-15 baseline + per-cluster evaluation helpers
SUMMARY_SENTENCES = 15

def lead15_from_doc(doc):
    sents = [norm_space(sent.text) for sent in doc.sents if norm_space(sent.text)]
    return " ".join(sents[:SUMMARY_SENTENCES])

def evaluate_candidate(candidate_doc, original_features):
    cand_entities = extract_entities(candidate_doc)
    cand_concepts = extract_concepts(candidate_doc)
    cand_triples = extract_triples(candidate_doc)

    entity_stats = prf_set(
        cand_entities.keys(),
        original_features["entities"].keys(),
    )
    concept_stats = prf_set(
        cand_concepts.keys(),
        original_features["concepts"].keys(),
    )
    relation_stats = prf_set(
        relation_keys(cand_triples),
        relation_keys(original_features["triples"]),
    )
    exact_stats = prf_set(
        exact_triple_keys(cand_triples),
        exact_triple_keys(original_features["triples"]),
    )
    soft_stats = soft_triple_prf(
        cand_triples,
        original_features["triples"],
    )
    dr_stats = domain_range_consistency(
        cand_triples,
        original_features["triples"],
    )
    struct_stats = structural_metrics(
        cand_triples,
        original_features["triples"],
        top_k=10,
    )

    return {
        "concept_precision": concept_stats["precision"],
        "concept_recall": concept_stats["recall"],
        "concept_f1": concept_stats["f1"],
        "concept_jaccard": concept_stats["jaccard"],

        "entity_precision": entity_stats["precision"],
        "entity_recall": entity_stats["recall"],
        "entity_f1": entity_stats["f1"],
        "entity_jaccard": entity_stats["jaccard"],

        "relation_precision": relation_stats["precision"],
        "relation_recall": relation_stats["recall"],
        "relation_f1": relation_stats["f1"],
        "relation_jaccard": relation_stats["jaccard"],

        "exact_triple_precision": exact_stats["precision"],
        "exact_triple_recall": exact_stats["recall"],
        "exact_triple_f1": exact_stats["f1"],
        "exact_triple_jaccard": exact_stats["jaccard"],

        "soft_triple_precision": soft_stats["precision"],
        "soft_triple_recall": soft_stats["recall"],
        "soft_triple_f1": soft_stats["f1"],

        "domain_accuracy": dr_stats["domain_accuracy"],
        "range_accuracy": dr_stats["range_accuracy"],
        "domain_range_pair_accuracy": dr_stats["pair_accuracy"],
        "schema_violation_rate": dr_stats["violation_rate"],

        "node_retention": struct_stats["node_retention"],
        "edge_retention": struct_stats["edge_retention"],
        "edge_jaccard": struct_stats["edge_jaccard"],
        "relation_type_retention": struct_stats["relation_type_retention"],
        "lcc_ratio": struct_stats["lcc_ratio"],
        "central_node_preservation_at10": struct_stats["central_node_preservation_at10"],

        "candidate_concepts": int(len(cand_concepts)),
        "candidate_entities": int(len(cand_entities)),
        "candidate_triples": int(len(cand_triples)),
        "candidate_nodes": struct_stats["candidate_nodes"],
        "candidate_edges": struct_stats["candidate_edges"],
    }

print("Round-3 evaluation helpers ready.")

# Exact fair-budget Lead-W builder

This function is embedded from the supplied Round-3B v3.1 notebook.

In [ ]:
#@title 11. Exact Round-3B Lead-W builder
def closest_lead_word_budget(doc, target_words):
    # Select the leading-sentence prefix whose word count is closest to
    # target_words. If two prefixes are equally close, prefer the one that
    # does not exceed the target.
    target_words = max(1, int(target_words))
    sents = [norm_space(sent.text) for sent in doc.sents if norm_space(sent.text)]

    if not sents:
        return "", 0, 0, -target_words

    counts = [len(s.split()) for s in sents]
    cumulative = np.cumsum(counts)
    candidates = []

    for i, words in enumerate(cumulative, start=1):
        delta = int(words) - target_words
        candidates.append((
            abs(delta),
            1 if delta > 0 else 0,
            int(words),
            i,
            delta,
        ))
        if words > target_words:
            break

    candidates.sort()
    _, _, actual_words, n_sentences, delta = candidates[0]
    summary = " ".join(sents[:n_sentences])

    return summary, int(actual_words), int(n_sentences), int(delta)

In [ ]:
#@title 12. Reconstruct Lead-W text and verify against the definitive Round-3B artifact
lead_rows = []
for _, row in tqdm(CACHE.iterrows(), total=len(CACHE), desc="Reconstruct Lead-W"):
    doc = nlp(row["source_text"])
    text, words, nsent, delta = closest_lead_word_budget(doc, int(row["run82_words"]))
    lead_rows.append({
        "cluster_id": row["cluster_id"],
        "leadw_text": text,
        "leadw_words_rebuilt": words,
        "leadw_sentences_rebuilt": nsent,
        "leadw_delta_rebuilt": delta,
    })

LEAD = pd.DataFrame(lead_rows)
CACHE = CACHE.merge(LEAD, on="cluster_id", how="left", validate="one_to_one")

checks = pd.DataFrame({
    "word_match": CACHE["leadw_words_rebuilt"] == CACHE["leadw_words"],
    "sentence_match": CACHE["leadw_sentences_rebuilt"] == CACHE["leadw_sentences"],
})
display(checks.mean().rename("match_fraction"))

if not checks.all().all():
    bad = CACHE.loc[
        ~(checks["word_match"] & checks["sentence_match"]),
        ["cluster_id","leadw_words","leadw_words_rebuilt","leadw_sentences","leadw_sentences_rebuilt"]
    ]
    bad.to_csv(OUT / "leadw_parity_failures.csv", index=False)
    raise RuntimeError(
        "Lead-W reconstruction failed parity. See leadw_parity_failures.csv. "
        "Do not continue with robustness experiments."
    )

CACHE[[
    "cluster_id","raw_index","source_text","run82_text","leadw_text",
    "original_words","run82_words","leadw_words","compression_gain"
]].to_csv(OUT / "paper2a_text_cache.csv", index=False)

print("PASS: rebuilt Lead-W text matches definitive Round-3B word and sentence counts for all clusters.")

In [ ]:
#@title 13. Statistical helpers — paired bootstrap, Wilcoxon, Cohen's dz
def paired_statistics(a, b, n_boot=10000, seed=SEED_ROBUSTNESS):
    A = pd.to_numeric(pd.Series(a), errors="coerce").to_numpy(float)
    B = pd.to_numeric(pd.Series(b), errors="coerce").to_numpy(float)
    mask = np.isfinite(A) & np.isfinite(B)
    A = A[mask]
    B = B[mask]
    d = A - B

    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    chunk = 500
    for start in range(0, n_boot, chunk):
        k = min(chunk, n_boot-start)
        idx = rng.integers(0, len(d), size=(k, len(d)))
        boot[start:start+k] = d[idx].mean(axis=1)

    if np.allclose(d, 0):
        wstat, pval = 0.0, 1.0
    else:
        w = wilcoxon(d, zero_method="wilcox", alternative="two-sided", method="auto")
        wstat, pval = float(w.statistic), float(w.pvalue)

    sd = d.std(ddof=1)
    return {
        "n": len(d),
        "a_mean": float(A.mean()),
        "b_mean": float(B.mean()),
        "mean_delta": float(d.mean()),
        "median_delta": float(np.median(d)),
        "ci95_low": float(np.percentile(boot, 2.5)),
        "ci95_high": float(np.percentile(boot, 97.5)),
        "wilcoxon_statistic": wstat,
        "wilcoxon_p_value": pval,
        "cohens_dz": float(d.mean()/sd) if sd > 0 else 0.0,
    }

def stratified_ids(frame, n, seed=SEED_ROBUSTNESS):
    x = frame[["cluster_id","compression_gain"]].copy()
    x["quartile"] = pd.qcut(x["compression_gain"], 4, labels=False, duplicates="drop")
    if not n or n >= len(x):
        return x["cluster_id"].tolist()

    per = n // 4
    parts = []
    for q, g in x.groupby("quartile"):
        parts.append(g.sample(n=min(per, len(g)), random_state=seed+int(q)))
    sel = pd.concat(parts)
    if len(sel) < n:
        pool = x[~x.cluster_id.isin(sel.cluster_id)]
        sel = pd.concat([
            sel,
            pool.sample(n=n-len(sel), random_state=seed+999)
        ])
    return sel["cluster_id"].tolist()

# TASK 1 — Modern fair-budget baseline: PRIMERA

**Design**

- Prespecified stratified sample from the Multi-News TEST split.
- Selection depends only on Run82 compression-gain quartile, not on preservation outcomes.
- PRIMERA is generated first, then post-hoc budgeted to the **same Run82 word budget**.
- Primary comparison uses exact word-budget output.
- Sentence-boundary budgeted PRIMERA is retained as a sensitivity analysis.
- All structured metrics use the **same authoritative Round-3 evaluator** as Run82 and Lead-W.

Set `RUN_PRIMERA=True` to execute. Results checkpoint after every cluster.

In [ ]:
#@title 14. Freeze the modern-baseline subset
MODERN_IDS = stratified_ids(CACHE, MODERN_BASELINE_N, SEED_ROBUSTNESS)
PRIMERA_INPUT = CACHE[CACHE.cluster_id.isin(MODERN_IDS)].copy().sort_values("cluster_id")
PRIMERA_INPUT[["cluster_id","compression_gain"]].to_csv(
    OUT / "10_primera_prespecified_ids.csv", index=False
)
print("PRIMERA N =", len(PRIMERA_INPUT))
display(pd.qcut(PRIMERA_INPUT["compression_gain"], 4, labels=False, duplicates="drop").value_counts().sort_index())

In [ ]:
#@title 15. Generate PRIMERA at the Run82 word budget
RUN_PRIMERA = True  #@param {type:"boolean"}
PRIMERA_MODEL = "allenai/PRIMERA-multinews"  #@param {type:"string"}

def nwords(text):
    return len(re.findall(r"\S+", str(text)))

def exact_word_budget(text, budget):
    return " ".join(re.findall(r"\S+", str(text))[:int(budget)])

def sentence_boundary_budget(text, budget):
    doc = nlp(str(text))
    sents = [norm_space(s.text) for s in doc.sents if norm_space(s.text)]
    out = []
    for s in sents:
        candidate = " ".join(out + [s]).strip()
        if nwords(candidate) <= int(budget):
            out.append(s)
        else:
            break
    return " ".join(out).strip()

PRIMERA_CKPT = CKPT / "primera_generated.csv"

if RUN_PRIMERA:
    from transformers import AutoTokenizer, LEDForConditionalGeneration

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(PRIMERA_MODEL)
    model = LEDForConditionalGeneration.from_pretrained(PRIMERA_MODEL).to(device).eval()

    model_commit = getattr(model.config, "_commit_hash", None)
    print("PRIMERA model:", PRIMERA_MODEL)
    print("Resolved commit:", model_commit)
    print("Device:", device)

    if PRIMERA_CKPT.exists():
        pr_rows = pd.read_csv(PRIMERA_CKPT).to_dict("records")
        done = set(pd.read_csv(PRIMERA_CKPT)["cluster_id"].astype(str))
    else:
        pr_rows, done = [], set()

    for _, row in tqdm(PRIMERA_INPUT.iterrows(), total=len(PRIMERA_INPUT), desc="PRIMERA"):
        cid = str(row["cluster_id"])
        if cid in done:
            continue

        budget = int(row["run82_words"])
        source = str(raw_test[int(row["raw_index"])]["document"]).replace("|||||", " <doc-sep> ")
        enc = tok(source, return_tensors="pt", truncation=True, max_length=4096)
        enc = {k:v.to(device) for k,v in enc.items()}

        min_new = max(32, int(budget * 1.10))
        max_new = max(min_new + 16, int(budget * 1.80))

        with torch.inference_mode():
            ids = model.generate(
                **enc,
                num_beams=5,
                min_new_tokens=min_new,
                max_new_tokens=max_new,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )

        raw = tok.decode(ids[0], skip_special_tokens=True).strip()
        exact = exact_word_budget(raw, budget)
        sentb = sentence_boundary_budget(raw, budget)

        pr_rows.append({
            "cluster_id": cid,
            "target_run82_words": budget,
            "primera_raw": raw,
            "primera_raw_words": nwords(raw),
            "primera_exact_word_budgeted": exact,
            "primera_exact_words": nwords(exact),
            "primera_sentence_budgeted": sentb,
            "primera_sentence_words": nwords(sentb),
            "primera_sentence_mismatch_pct": abs(nwords(sentb)-budget)/max(1,budget)*100,
            "model_id": PRIMERA_MODEL,
            "model_commit": model_commit,
        })
        done.add(cid)
        pd.DataFrame(pr_rows).to_csv(PRIMERA_CKPT, index=False)

    PRIMERA = pd.DataFrame(pr_rows).drop_duplicates("cluster_id", keep="last")
    if len(PRIMERA) != len(PRIMERA_INPUT):
        raise RuntimeError(f"PRIMERA incomplete: {len(PRIMERA)}/{len(PRIMERA_INPUT)}")
    PRIMERA.to_csv(OUT / "11_primera_generated.csv", index=False)
    display(PRIMERA[[
        "target_run82_words","primera_raw_words","primera_exact_words",
        "primera_sentence_words","primera_sentence_mismatch_pct"
    ]].describe())
else:
    print("PRIMERA generation skipped.")

In [ ]:
#@title 16. Evaluate PRIMERA with the authoritative Round-3 preservation pipeline
if RUN_PRIMERA:
    PRIMERA = pd.read_csv(OUT / "11_primera_generated.csv")
    work = PRIMERA_INPUT[["cluster_id","source_text"]].merge(
        PRIMERA, on="cluster_id", how="inner", validate="one_to_one"
    )

    eval_rows = []
    for _, row in tqdm(work.iterrows(), total=len(work), desc="Evaluate PRIMERA"):
        original_doc = nlp(row["source_text"])
        original_features = {
            "entities": extract_entities(original_doc),
            "concepts": extract_concepts(original_doc),
            "triples": extract_triples(original_doc),
        }

        for variant, text in [
            ("primera_exact", row["primera_exact_word_budgeted"]),
            ("primera_sentence", row["primera_sentence_budgeted"]),
        ]:
            metrics = evaluate_candidate(nlp(str(text)), original_features)
            eval_rows.append({
                "cluster_id": row["cluster_id"],
                "system": variant,
                **metrics,
            })

    PRIMERA_METRICS = pd.DataFrame(eval_rows)
    PRIMERA_METRICS.to_csv(OUT / "12_primera_structured_metrics.csv", index=False)

    # Compare exact-budget PRIMERA with Run82 and Lead-W on exactly the same IDs.
    pm = PRIMERA_METRICS[PRIMERA_METRICS.system=="primera_exact"].set_index("cluster_id")
    run = R3[R3.cluster_id.isin(pm.index)].set_index("cluster_id")
    lead = R3B[R3B.cluster_id.isin(pm.index)].set_index("cluster_id")

    structured_metrics = [
        "concept_f1",
        "relation_f1",
        "exact_triple_f1",
        "soft_triple_f1",
        "domain_range_pair_accuracy",
        "edge_retention",
    ]
    rows = []
    for metric in structured_metrics:
        for comparator, comp_series in [
            ("Run82", run[f"run82_{metric}"]),
            ("Lead-W", lead[f"leadw_{metric}"]),
        ]:
            st = paired_statistics(pm[metric], comp_series.loc[pm.index])
            rows.append({
                "metric": metric,
                "comparison": f"PRIMERA_exact_minus_{comparator}",
                "primera_mean": st["a_mean"],
                "comparator_mean": st["b_mean"],
                "delta": st["mean_delta"],
                "ci95_low": st["ci95_low"],
                "ci95_high": st["ci95_high"],
                "wilcoxon_p": st["wilcoxon_p_value"],
                "cohens_dz": st["cohens_dz"],
                "n": st["n"],
            })

    PRIMERA_COMPARISON = pd.DataFrame(rows)
    PRIMERA_COMPARISON.to_csv(OUT / "13_primera_fair_budget_comparison.csv", index=False)
    display(PRIMERA_COMPARISON.round(6))

# TASK 2 — Alternative extractor robustness: REBEL

The REBEL subset is prespecified independently of preservation outcomes by Run82 compression-gain quartile.

Primary systems:
- Run82
- Lead-W

If PRIMERA has been generated for overlapping IDs, it is included as an optional third system.

The robustness question is **not** whether REBEL reproduces the same absolute scores. It is whether the component-versus-composition pattern remains qualitatively stable under an independent relation extractor.

In [ ]:
#@title 17. Freeze the REBEL subset
REBEL_IDS = stratified_ids(CACHE, REBEL_N, SEED_ROBUSTNESS + 100)
REBEL_INPUT = CACHE[CACHE.cluster_id.isin(REBEL_IDS)][
    ["cluster_id","source_text","run82_text","leadw_text","compression_gain"]
].copy().sort_values("cluster_id")

if (OUT / "11_primera_generated.csv").exists():
    p = pd.read_csv(OUT / "11_primera_generated.csv")[
        ["cluster_id","primera_exact_word_budgeted"]
    ].rename(columns={"primera_exact_word_budgeted":"primera_text"})
    REBEL_INPUT = REBEL_INPUT.merge(p, on="cluster_id", how="left")

REBEL_INPUT.to_csv(OUT / "20_rebel_prespecified_input.csv", index=False)
print("REBEL N =", len(REBEL_INPUT))

In [ ]:
#@title 18. Run REBEL alternative-extractor robustness
RUN_REBEL = True  #@param {type:"boolean"}
REBEL_MODEL = "Babelscape/rebel-large"  #@param {type:"string"}

def rebel_norm(s):
    s = re.sub(r"\s+", " ", str(s).strip().lower())
    s = re.sub(r"^[^\w]+|[^\w]+$", "", s)
    return s

def parse_rebel(text):
    text = text.replace("<s>", "").replace("</s>", "").strip()
    triples = []
    for part in text.split("<triplet>"):
        part = part.strip()
        if not part or "<subj>" not in part or "<obj>" not in part:
            continue
        try:
            subj, rest = part.split("<subj>", 1)
            obj, rel = rest.split("<obj>", 1)
            t = (rebel_norm(subj), rebel_norm(rel), rebel_norm(obj))
            if all(t):
                triples.append(t)
        except ValueError:
            pass
    return list(dict.fromkeys(triples))

def rebel_reprs(triples):
    ents, rels, pairs, exact = set(), set(), set(), set()
    for s, r, o in triples:
        ents.update([s,o])
        rels.add(r)
        pairs.add((s,o))
        exact.add((s,r,o))
    return ents, rels, pairs, exact

def set_f1(ref, pred):
    ref, pred = set(ref), set(pred)
    if not ref and not pred:
        return 1.0
    if not ref or not pred:
        return 0.0
    inter = len(ref & pred)
    p = inter / len(pred)
    r = inter / len(ref)
    return 0.0 if p+r == 0 else 2*p*r/(p+r)

def rebel_chunks(text, max_chars=5000):
    text = str(text)
    if len(text) <= max_chars:
        return [text]
    sents = re.split(r"(?<=[.!?])\s+", text)
    out, cur = [], []
    for s in sents:
        if sum(len(x)+1 for x in cur) + len(s) > max_chars and cur:
            out.append(" ".join(cur))
            cur = [s]
        else:
            cur.append(s)
    if cur:
        out.append(" ".join(cur))
    return out

if RUN_REBEL:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    device = "cuda" if torch.cuda.is_available() else "cpu"
    rtok = AutoTokenizer.from_pretrained(REBEL_MODEL)
    rmodel = AutoModelForSeq2SeqLM.from_pretrained(REBEL_MODEL).to(device).eval()
    rebel_commit = getattr(rmodel.config, "_commit_hash", None)

    def rebel_extract(text):
        all_t = []
        for ch in rebel_chunks(text):
            enc = rtok(ch, return_tensors="pt", truncation=True, max_length=512).to(device)
            with torch.inference_mode():
                ids = rmodel.generate(**enc, max_new_tokens=256, num_beams=3)
            dec = rtok.decode(ids[0], skip_special_tokens=False)
            all_t.extend(parse_rebel(dec))
        return list(dict.fromkeys(all_t))

    ck = CKPT / "rebel_metrics.csv"
    if ck.exists():
        rebel_rows = pd.read_csv(ck).to_dict("records")
        done = set(zip(pd.read_csv(ck)["cluster_id"].astype(str), pd.read_csv(ck)["system"].astype(str)))
    else:
        rebel_rows, done = [], set()

    system_cols = ["run82_text","leadw_text"]
    if "primera_text" in REBEL_INPUT.columns and REBEL_INPUT["primera_text"].notna().any():
        system_cols.append("primera_text")

    for _, row in tqdm(REBEL_INPUT.iterrows(), total=len(REBEL_INPUT), desc="REBEL"):
        src_triples = rebel_extract(row["source_text"])
        src_repr = rebel_reprs(src_triples)

        for syscol in system_cols:
            key = (str(row["cluster_id"]), syscol)
            if key in done:
                continue
            text = row.get(syscol)
            if pd.isna(text):
                continue
            sm_triples = rebel_extract(text)
            sm_repr = rebel_reprs(sm_triples)
            rebel_rows.append({
                "cluster_id": row["cluster_id"],
                "system": syscol,
                "source_rebel_triples": len(src_triples),
                "summary_rebel_triples": len(sm_triples),
                "entity_f1": set_f1(src_repr[0], sm_repr[0]),
                "relation_f1": set_f1(src_repr[1], sm_repr[1]),
                "pair_f1": set_f1(src_repr[2], sm_repr[2]),
                "exact_triple_f1": set_f1(src_repr[3], sm_repr[3]),
                "model_id": REBEL_MODEL,
                "model_commit": rebel_commit,
            })
            done.add(key)
            pd.DataFrame(rebel_rows).to_csv(ck, index=False)

    REBEL = pd.DataFrame(rebel_rows)
    REBEL.to_csv(OUT / "21_rebel_metrics.csv", index=False)

    # Primary Run82 vs Lead-W paired statistics.
    wide = REBEL[REBEL.system.isin(["run82_text","leadw_text"])].pivot(
        index="cluster_id", columns="system"
    )
    rows = []
    for metric in ["entity_f1","relation_f1","pair_f1","exact_triple_f1"]:
        st = paired_statistics(
            wide[metric]["run82_text"],
            wide[metric]["leadw_text"],
        )
        rows.append({
            "metric": metric,
            "run82_mean": st["a_mean"],
            "leadw_mean": st["b_mean"],
            "delta": st["mean_delta"],
            "ci95_low": st["ci95_low"],
            "ci95_high": st["ci95_high"],
            "wilcoxon_p": st["wilcoxon_p_value"],
            "cohens_dz": st["cohens_dz"],
            "n": st["n"],
        })

    REBEL_SUMMARY = pd.DataFrame(rows)
    REBEL_SUMMARY.to_csv(OUT / "22_rebel_run82_vs_leadw_summary.csv", index=False)
    display(REBEL_SUMMARY.round(6))
else:
    print("REBEL skipped.")

# TASK 3 — True WebNLG alternative regrouping

This is a **true regrouping experiment**, not a shuffle of already-computed bundle metrics.

For each repetition:

1. shuffle the unique gold-triple sets with a fixed robustness-analysis RNG;
2. form new complete bundles of 25;
3. rerun frozen Run82 on each new bundle;
4. rebuild matched Lead-W from the new bundle;
5. run the same dependency extractor;
6. apply the same WebNLG literal-coverage and semantic gold matching;
7. aggregate Run82-minus-Lead-W effects.

The original deterministic grouping remains the confirmatory analysis; this section is a additional robustness analysis.

In [ ]:
#@title 19. Load WebNLG unique gold-triple sets using the Round-3C data contract
from sentence_transformers import SentenceTransformer

WEBNLG_URL = "https://huggingface.co/api/datasets/GEM/web_nlg/parquet/en/test/0.parquet"
WEBNLG_PARQUET = ROOT / "webnlg_en_test_0.parquet"

if not WEBNLG_PARQUET.exists():
    req = urllib.request.Request(WEBNLG_URL, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as response:
        WEBNLG_PARQUET.write_bytes(response.read())

b = WEBNLG_PARQUET.read_bytes()
if b[:4] != b"PAR1" or b[-4:] != b"PAR1":
    raise RuntimeError("Invalid WebNLG parquet download.")

raw_pdf = pd.read_parquet(WEBNLG_PARQUET)

def norm_ws(x):
    return re.sub(r"\s+", " ", str(x)).strip()

def triple_key_list(inp):
    vals = []
    for x in inp:
        x = norm_ws(x)
        parts = [norm_ws(p) for p in x.split("|")]
        if len(parts) == 3:
            vals.append(tuple(parts))
    return tuple(vals)

rows = []
for i, row in raw_pdf.iterrows():
    inp = row["input"]
    target = norm_ws(row["target"])
    if inp is None or len(inp) == 0 or not target:
        continue
    gold = triple_key_list(inp)
    if not gold:
        continue
    parent = row.get("gem_parent_id") or row.get("webnlg_id") or f"row-{i}"
    rows.append({
        "row_index": int(i),
        "parent_id": str(parent),
        "webnlg_id": str(row.get("webnlg_id","")),
        "category": str(row.get("category","UNKNOWN")),
        "target": target,
        "gold_triples": gold,
        "n_gold_triples": len(gold),
    })

WEBNLG_SETS = (
    pd.DataFrame(rows)
    .sort_values(["parent_id","row_index"], kind="stable")
    .drop_duplicates("parent_id", keep="first")
    .reset_index(drop=True)
)

print("Unique gold-triple sets:", len(WEBNLG_SETS))
print("Full bundles at 25 sets:", len(WEBNLG_SETS)//25)

### Patch: missing Round-3C dependency extractor

The first FINAL notebook omitted the Round-3C `extract_dependency_triples()` utility even though Cell 21 calls it.  
The cell below restores the exact dependency extractor used by the authoritative Round-3C v3.2.6 pipeline.

In [ ]:
#@title FIX — Round-3C dependency extractor required by true WebNLG regrouping
# This cell is copied from the authoritative Round-3C v3.2.6 notebook.
# It must run before Cell 21 (Repeated true WebNLG regrouping).

def normalize_label(value):
    value = clean_text(value).lower()
    value = re.sub(r"[^a-z0-9_\s-]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def subtree_text(token):
    return clean_text(
        " ".join(
            t.text
            for t in sorted(token.subtree, key=lambda x: x.i)
        )
    )

def extract_dependency_triples(text):
    text = clean_text(text)
    if not text:
        return []

    doc = nlp_dep(text)
    candidates = []

    for sentence in doc.sents:
        roots = [
            token
            for token in sentence
            if token.dep_ == "ROOT"
            and token.pos_ in {"VERB", "AUX"}
        ]

        for root in roots:
            children = list(root.children)

            subjects = [
                token
                for token in children
                if token.dep_ in {"nsubj", "nsubjpass", "csubj"}
            ]

            objects = [
                token
                for token in children
                if token.dep_ in {"dobj", "obj", "attr", "oprd", "dative"}
            ]

            for subject in subjects:
                for obj in objects:
                    candidates.append((
                        subtree_text(subject),
                        root.lemma_ or root.text,
                        subtree_text(obj),
                        sentence.text,
                    ))

            for prep in [
                token for token in children if token.dep_ == "prep"
            ]:
                prep_objects = [
                    token
                    for token in prep.children
                    if token.dep_ in {"pobj", "obj"}
                ]
                for subject in subjects:
                    for obj in prep_objects:
                        predicate = (
                            f"{root.lemma_ or root.text}_"
                            f"{prep.lemma_ or prep.text}"
                        )
                        candidates.append((
                            subtree_text(subject),
                            predicate,
                            subtree_text(obj),
                            sentence.text,
                        ))

    output = []
    seen = set()

    for subject, predicate, obj, evidence in candidates:
        key = (
            normalize_label(subject),
            normalize_label(predicate),
            normalize_label(obj),
        )
        if all(key) and key not in seen:
            seen.add(key)
            output.append({
                "subject": clean_text(subject),
                "predicate": clean_text(predicate),
                "object": clean_text(obj),
                "evidence": clean_text(evidence),
                "method": "dependency",
            })

    return output

# Hard guard so the missing-definition error cannot recur.
assert callable(extract_dependency_triples)
_test_dep = extract_dependency_triples("Alice founded Acme in London.")
assert isinstance(_test_dep, list)
print("PASS: Round-3C dependency extractor is defined and ready.")

In [ ]:
#@title 20. Original Round-3C gold normalization and semantic matching
sem_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Semantic matcher commit:", getattr(sem_model, "_model_card_vars", {}).get("model_revision", "runtime-resolved"))

#@title 7. Gold normalization, Lead-W, and semantic triple matcher
SEM_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

def clean_gold_term(x):
    x = str(x).strip().strip('"').replace('_',' ')
    # Split camelCase / PascalCase for DBpedia properties.
    x = re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', x)
    x = re.sub(r'\s+', ' ', x).lower().strip()
    return x

def gold_norm_triple(t):
    s,r,o = t
    return (
        clean_gold_term(s),
        clean_gold_term(r),
        clean_gold_term(o),
    )

def text_tokens(x):
    return set(re.findall(r'[a-z0-9]+', clean_gold_term(x)))

def literal_mention(term, text):
    toks = text_tokens(term)
    if not toks:
        return False
    text_toks = text_tokens(text)
    # Allow all informative tokens of DBpedia labels to occur in the output.
    return toks.issubset(text_toks)

def lead_w_from_documents(documents, target_words):
    text = ' '.join(documents)
    doc = nlp_dep(text)
    sents = [clean_text(s.text) for s in doc.sents if clean_text(s.text)]
    target_words = max(1,int(target_words))

    candidates=[]
    total=0
    for i,s in enumerate(sents, start=1):
        total += len(s.split())
        delta = total-target_words
        candidates.append((abs(delta), 1 if delta>0 else 0, total, i, delta))
        if total>target_words:
            break

    if not candidates:
        return '',0,0,0

    candidates.sort()
    _,_,words,n_sents,delta = candidates[0]
    return ' '.join(sents[:n_sents]), int(words), int(n_sents), int(delta)

def dep_to_tuple(t):
    return (
        clean_gold_term(t.get('subject','')),
        clean_gold_term(t.get('predicate','')),
        clean_gold_term(t.get('object','')),
    )

def component_embeddings(strings):
    if not strings:
        return np.empty((0,384),dtype=np.float32)
    return sem_model.encode(
        list(strings),
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=128,
        show_progress_bar=False,
    )

def semantic_gold_match(gold_triples, extracted_triples, threshold=0.60):
    gold = [gold_norm_triple(t) for t in gold_triples]
    ext = [dep_to_tuple(t) for t in extracted_triples]

    if not gold:
        return {'matches':0,'precision':0.0,'recall':0.0,'f1':0.0}
    if not ext:
        return {'matches':0,'precision':0.0,'recall':0.0,'f1':0.0}

    gold_s = component_embeddings([x[0] for x in gold])
    gold_r = component_embeddings([x[1] for x in gold])
    gold_o = component_embeddings([x[2] for x in gold])

    ext_s = component_embeddings([x[0] for x in ext])
    ext_r = component_embeddings([x[1] for x in ext])
    ext_o = component_embeddings([x[2] for x in ext])

    s_sim = np.asarray(gold_s @ ext_s.T)
    r_sim = np.asarray(gold_r @ ext_r.T)
    o_sim = np.asarray(gold_o @ ext_o.T)

    # Component-weighted semantic score.
    score = 0.35*s_sim + 0.30*r_sim + 0.35*o_sim

    pairs=[]
    for i in range(score.shape[0]):
        for j in range(score.shape[1]):
            # Guard against relation-only or entity-only accidental matches.
            if s_sim[i,j] < 0.45 or o_sim[i,j] < 0.45 or r_sim[i,j] < 0.25:
                continue
            if score[i,j] >= threshold:
                pairs.append((float(score[i,j]),i,j))

    pairs.sort(key=lambda x:(-x[0],x[1],x[2]))
    used_g=set()
    used_e=set()
    matches=0

    for sc,i,j in pairs:
        if i in used_g or j in used_e:
            continue
        used_g.add(i)
        used_e.add(j)
        matches += 1

    p = matches/len(ext) if ext else 0.0
    r = matches/len(gold) if gold else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0

    return {
        'matches':int(matches),
        'precision':float(p),
        'recall':float(r),
        'f1':float(f),
    }

def literal_gold_coverage(gold_triples, text):
    gold=[gold_norm_triple(t) for t in gold_triples]
    if not gold:
        return {
            'subject_coverage':0.0,'object_coverage':0.0,
            'entity_pair_coverage':0.0,'predicate_lexical_coverage':0.0,
        }

    sub = [literal_mention(s,text) for s,_,_ in gold]
    obj = [literal_mention(o,text) for _,_,o in gold]
    pair = [a and b for a,b in zip(sub,obj)]
    pred = [literal_mention(r,text) for _,r,_ in gold]

    return {
        'subject_coverage':float(np.mean(sub)),
        'object_coverage':float(np.mean(obj)),
        'entity_pair_coverage':float(np.mean(pair)),
        'predicate_lexical_coverage':float(np.mean(pred)),
    }

print('Gold semantic matcher ready:', SEM_MODEL_NAME)

In [ ]:
#@title 21. Repeated true WebNLG regrouping
RUN_WEBNLG_REGROUPING = True  #@param {type:"boolean"}

WEB_METRICS = [
    "subject_coverage",
    "object_coverage",
    "entity_pair_coverage",
    "predicate_lexical_coverage",
    "gold_semantic_recall_t060",
    "gold_semantic_f1_t060",
]

def build_bundle(chunk, rep, bundle_no):
    gold = []
    cats = []
    for rec in chunk.to_dict("records"):
        gold.extend(list(rec["gold_triples"]))
        cats.append(rec["category"])
    return {
        "bundle_id": f"rep{rep:03d}_bundle{bundle_no:03d}",
        "documents": chunk["target"].tolist(),
        "gold_triples": gold,
        "n_documents": len(chunk),
        "n_gold_triples": len(gold),
        "categories": cats,
    }

if RUN_WEBNLG_REGROUPING:
    regroup_ckpt = CKPT / "webnlg_regrouping_summary.csv"
    if regroup_ckpt.exists():
        rep_rows = pd.read_csv(regroup_ckpt).to_dict("records")
        completed_reps = set(pd.read_csv(regroup_ckpt)["regrouping"].astype(int))
    else:
        rep_rows, completed_reps = [], set()

    n_bundles = len(WEBNLG_SETS) // 25
    n_used = n_bundles * 25

    for rep in tqdm(range(WEBNLG_REGROUPINGS), desc="WebNLG regroupings"):
        if rep in completed_reps:
            continue

        rng = np.random.default_rng(SEED_ROBUSTNESS + rep)
        perm = rng.permutation(len(WEBNLG_SETS))
        ordered = WEBNLG_SETS.iloc[perm[:n_used]].reset_index(drop=True)

        bundle_records = []
        for bno in range(n_bundles):
            chunk = ordered.iloc[bno*25:(bno+1)*25]
            bundle = build_bundle(chunk, rep, bno+1)
            docs = bundle["documents"]
            gold = bundle["gold_triples"]
            source_text = " ".join(docs)

            run82_summary, _ = generate_summary(docs)
            run82_words = len(run82_summary.split())
            leadw_summary, leadw_words, _, _ = lead_w_from_documents(docs, run82_words)

            run82_dep = extract_dependency_triples(run82_summary)
            leadw_dep = extract_dependency_triples(leadw_summary)

            rec = {"bundle_id":bundle["bundle_id"]}

            for prefix, text in [("run82",run82_summary),("leadw",leadw_summary)]:
                cov = literal_gold_coverage(gold, text)
                for k,v in cov.items():
                    rec[f"{prefix}_{k}"] = v

            for prefix, dep in [("run82",run82_dep),("leadw",leadw_dep)]:
                m = semantic_gold_match(gold, dep, threshold=0.60)
                rec[f"{prefix}_gold_semantic_recall_t060"] = m["recall"]
                rec[f"{prefix}_gold_semantic_f1_t060"] = m["f1"]

            bundle_records.append(rec)

        B = pd.DataFrame(bundle_records)
        for metric in WEB_METRICS:
            st = paired_statistics(
                B[f"run82_{metric}"],
                B[f"leadw_{metric}"],
                n_boot=3000,
                seed=SEED_ROBUSTNESS + rep,
            )
            rep_rows.append({
                "regrouping": rep,
                "metric": metric,
                "n_bundles": len(B),
                "run82_mean": st["a_mean"],
                "leadw_mean": st["b_mean"],
                "delta": st["mean_delta"],
                "ci95_low": st["ci95_low"],
                "ci95_high": st["ci95_high"],
                "wilcoxon_p": st["wilcoxon_p_value"],
                "cohens_dz": st["cohens_dz"],
            })

        completed_reps.add(rep)
        pd.DataFrame(rep_rows).to_csv(regroup_ckpt, index=False)
        gc.collect()

    WEB_REGROUP = pd.DataFrame(rep_rows)
    WEB_REGROUP.to_csv(OUT / "31_webnlg_true_regrouping_all.csv", index=False)

    WEB_REGROUP_SUMMARY = (
        WEB_REGROUP.groupby("metric")
        .agg(
            regroupings=("regrouping","nunique"),
            mean_delta=("delta","mean"),
            delta_q025=("delta",lambda x:np.quantile(x,.025)),
            delta_q975=("delta",lambda x:np.quantile(x,.975)),
            direction_positive_fraction=("delta",lambda x:float(np.mean(np.asarray(x)>0))),
            significant_fraction=("wilcoxon_p",lambda x:float(np.mean(np.asarray(x)<.05))),
            median_p=("wilcoxon_p","median"),
            median_dz=("cohens_dz","median"),
        )
        .reset_index()
    )
    WEB_REGROUP_SUMMARY.to_csv(OUT / "32_webnlg_true_regrouping_summary.csv", index=False)
    display(WEB_REGROUP_SUMMARY.round(6))
else:
    print("WebNLG regrouping skipped.")

# TASK 4 — Run82 mechanism ablation

Ablations are based on the original frozen summariser implementation.

Variants:

- **full_run82:** α=0.30, β=0.70, δ=1.00
- **no_topic:** α=0.00, β=1.00, δ=1.00
- **no_pattern:** α=1.00, β=0.00, δ=1.00
- **no_redundancy:** α=0.30, β=0.70, δ=0.00

The topic/pattern ablations renormalize the remaining relevance term to 1.0, matching the major-revision package's intended mechanism test.

Before any ablation is accepted, the parameterized wrapper must reproduce the stored definitive Run82 summaries on the prespecified subset.

In [ ]:
#@title 22. Parameterized wrapper around the authoritative frozen Run82 logic
def generate_summary_variant(
    documents,
    alpha_topic=0.30,
    beta_pattern=0.70,
    delta_redundancy=1.00,
    summary_budget=15,
    n_topics=25,
    seed=42,
):
    sentences = safe_sentencize(documents)
    if not sentences:
        return "", {"n_sentences":0,"patterns_found":0,"candidate_count":0}

    target = max(1, min(int(summary_budget), len(sentences)))
    if len(sentences) <= target:
        return " ".join(sentences), {
            "n_sentences":len(sentences),
            "patterns_found":0,
            "candidate_count":len(sentences),
        }

    topic_model = fit_topic_model(sentences, n_topics=n_topics, seed=seed)
    topic = np.asarray(topic_model["topic_score"], dtype=np.float32)
    patt, patterns = pattern_scores(
        sentences, topic_model=topic_model, min_support=MIN_PATTERN_SUPPORT
    )
    patt = np.asarray(patt, dtype=np.float32)

    candidates = build_candidate_pool(
        topic_model, per_topic=CANDIDATE_POOL_PER_TOPIC
    )
    if candidates.size == 0 or len(candidates) < target:
        candidates = np.arange(len(sentences), dtype=int)

    base = alpha_topic * topic + beta_pattern * patt
    ordered_candidates = sorted(
        (int(i) for i in candidates),
        key=lambda i:(-float(base[i]), int(i)),
    )

    sim, active_candidates = _paper1_similarity(sentences, ordered_candidates)
    active_candidates = list(active_candidates)
    position = {idx:pos for pos,idx in enumerate(active_candidates)}

    selected = []
    remaining = set(active_candidates)

    while remaining and len(selected) < target:
        best_idx = None
        best_score = -np.inf

        for idx in sorted(remaining):
            if selected:
                i = position[idx]
                redundancy = max(float(sim[i, position[j]]) for j in selected)
            else:
                redundancy = 0.0

            score = (
                alpha_topic * topic[idx]
                + beta_pattern * patt[idx]
                - delta_redundancy * redundancy
            )

            if (
                score > best_score
                or (
                    np.isclose(score, best_score)
                    and (best_idx is None or idx < best_idx)
                )
            ):
                best_score = score
                best_idx = int(idx)

        if best_idx is None:
            break

        selected.append(best_idx)
        remaining.remove(best_idx)

    summary = " ".join(sentences[i] for i in sorted(selected))
    return summary, {
        "n_sentences":len(sentences),
        "patterns_found":len(patterns),
        "candidate_count":len(active_candidates),
    }

In [ ]:
#@title 23. Freeze ablation subset and HARD parity guard against definitive Run82
ABLATION_IDS = stratified_ids(CACHE, ABLATION_N, SEED_ROBUSTNESS + 200)
ABLATION_INPUT = CACHE[CACHE.cluster_id.isin(ABLATION_IDS)].copy().sort_values("cluster_id")
ABLATION_INPUT[["cluster_id","compression_gain"]].to_csv(
    OUT / "40_ablation_prespecified_ids.csv", index=False
)

# HARD parity guard on the entire ablation subset.
parity_failures = []
for _, row in tqdm(ABLATION_INPUT.iterrows(), total=len(ABLATION_INPUT), desc="Run82 parity guard"):
    documents = SOURCE.loc[SOURCE.cluster_id==row["cluster_id"], "documents"].iloc[0]
    rebuilt, _ = generate_summary_variant(
        documents,
        alpha_topic=0.30,
        beta_pattern=0.70,
        delta_redundancy=1.00,
        summary_budget=15,
        n_topics=25,
        seed=42,
    )
    if clean_text(rebuilt) != clean_text(row["run82_text"]):
        parity_failures.append({
            "cluster_id": row["cluster_id"],
            "stored": row["run82_text"],
            "rebuilt": rebuilt,
        })

if parity_failures:
    pd.DataFrame(parity_failures).to_csv(OUT / "41_ablation_run82_parity_failures.csv", index=False)
    raise RuntimeError(
        f"Embedded Run82 wrapper failed exact text parity for {len(parity_failures)} clusters. "
        "Ablation is blocked."
    )

print("PASS: parameterized wrapper exactly reproduces stored definitive Run82 summaries on the ablation subset.")

In [ ]:
#@title 24. Execute topic/pattern/redundancy ablations with the same Round-3 evaluator
RUN_ABLATION = True  #@param {type:"boolean"}

ABLATION_CONFIGS = [
    ("full_run82",      0.30, 0.70, 1.00),
    ("no_topic",        0.00, 1.00, 1.00),
    ("no_pattern",      1.00, 0.00, 1.00),
    ("no_redundancy",   0.30, 0.70, 0.00),
]

if RUN_ABLATION:
    ck = CKPT / "ablation_per_cluster.csv"
    if ck.exists():
        ab_rows = pd.read_csv(ck).to_dict("records")
        done = set(zip(pd.read_csv(ck)["cluster_id"].astype(str), pd.read_csv(ck)["variant"].astype(str)))
    else:
        ab_rows, done = [], set()

    for _, row in tqdm(ABLATION_INPUT.iterrows(), total=len(ABLATION_INPUT), desc="Ablation clusters"):
        cid = str(row["cluster_id"])
        documents = SOURCE.loc[SOURCE.cluster_id==cid, "documents"].iloc[0]
        original_doc = nlp(row["source_text"])
        original_features = {
            "entities": extract_entities(original_doc),
            "concepts": extract_concepts(original_doc),
            "triples": extract_triples(original_doc),
        }

        for variant, alpha, beta, delta in ABLATION_CONFIGS:
            if (cid, variant) in done:
                continue

            summary, _ = generate_summary_variant(
                documents,
                alpha_topic=alpha,
                beta_pattern=beta,
                delta_redundancy=delta,
                summary_budget=15,
                n_topics=25,
                seed=42,
            )
            metrics = evaluate_candidate(nlp(summary), original_features)
            words = len(summary.split())
            target_words = int(row["run82_words"])
            ab_rows.append({
                "cluster_id": cid,
                "variant": variant,
                "alpha_topic": alpha,
                "beta_pattern": beta,
                "delta_redundancy": delta,
                "summary_words": words,
                "run82_target_words": target_words,
                "absolute_budget_mismatch_pct": abs(words-target_words)/max(1,target_words)*100,
                **metrics,
            })
            done.add((cid, variant))
            pd.DataFrame(ab_rows).to_csv(ck, index=False)

    ABLATION = pd.DataFrame(ab_rows)
    ABLATION.to_csv(OUT / "42_ablation_per_cluster.csv", index=False)

    core_metrics = [
        "concept_f1","relation_f1","exact_triple_f1","soft_triple_f1",
        "domain_range_pair_accuracy","edge_retention"
    ]
    means = ABLATION.groupby("variant")[core_metrics + ["summary_words","absolute_budget_mismatch_pct"]].mean().reset_index()
    means.to_csv(OUT / "43_ablation_means.csv", index=False)

    full = ABLATION[ABLATION.variant=="full_run82"].set_index("cluster_id")
    comp_rows = []
    for variant in ["no_topic","no_pattern","no_redundancy"]:
        alt = ABLATION[ABLATION.variant==variant].set_index("cluster_id")
        ids = full.index.intersection(alt.index)
        for metric in core_metrics:
            st = paired_statistics(full.loc[ids,metric], alt.loc[ids,metric])
            comp_rows.append({
                "ablation": variant,
                "metric": metric,
                "full_mean": st["a_mean"],
                "ablation_mean": st["b_mean"],
                "full_minus_ablation": st["mean_delta"],
                "ci95_low": st["ci95_low"],
                "ci95_high": st["ci95_high"],
                "wilcoxon_p": st["wilcoxon_p_value"],
                "cohens_dz": st["cohens_dz"],
                "n": st["n"],
            })

    ABLATION_COMPARISON = pd.DataFrame(comp_rows)
    ABLATION_COMPARISON.to_csv(OUT / "44_ablation_paired_comparisons.csv", index=False)

    # Strict word-budget sensitivity for each variant: only rows within ±5% of stored Run82 words.
    strict_rows = []
    for variant in [x[0] for x in ABLATION_CONFIGS]:
        g = ABLATION[
            (ABLATION.variant==variant) &
            (ABLATION.absolute_budget_mismatch_pct <= 5.0)
        ]
        for metric in core_metrics:
            strict_rows.append({
                "variant": variant,
                "metric": metric,
                "n": len(g),
                "mean": float(g[metric].mean()) if len(g) else np.nan,
            })
    ABLATION_STRICT = pd.DataFrame(strict_rows)
    ABLATION_STRICT.to_csv(OUT / "45_ablation_strict5pct_sensitivity.csv", index=False)

    display(means.round(6))
    display(ABLATION_COMPARISON.round(6))
else:
    print("Ablation skipped.")

# Final robustness tables and interpretation gates

This section creates one Excel workbook containing every table needed for the revised manuscript and point-by-point response.

The workbook includes:
- definitive Round 3B comparison;
- definitive Round 3C comparison;
- 0.3997 vs 0.3994 reconciliation;
- PRIMERA modern baseline;
- REBEL alternative-extractor robustness;
- WebNLG true regrouping;
- ablation means and paired tests;
- strict word-budget ablation sensitivity;
- completion/status matrix.

Interpretation is deliberately conservative. Statistical significance is never treated as practical importance automatically.

In [ ]:
#@title 25. Build robustness completion/status matrix
def exists(name):
    return (OUT / name).exists()

status_rows = [
    ["Definitive original evidence audit", "PASS", "Round2/3/3B/3C provenance verified"],
    ["0.3997 vs 0.3994 reconciliation", "PASS", "Distinct metric objects identified and documented"],
    ["Modern fair-budget baseline",
     "PASS" if exists("13_primera_fair_budget_comparison.csv") else "NOT RUN",
     "PRIMERA exact word-budget output evaluated with original Round-3 metrics"],
    ["Alternative extractor robustness",
     "PASS" if exists("22_rebel_run82_vs_leadw_summary.csv") else "NOT RUN",
     "REBEL independent extractor on outcome-independent stratified subset"],
    ["True WebNLG regrouping",
     "PASS" if exists("32_webnlg_true_regrouping_summary.csv") else "NOT RUN",
     "Alternative bundle construction reruns original bundle pipeline"],
    ["Run82 mechanism ablation",
     "PASS" if exists("44_ablation_paired_comparisons.csv") else "NOT RUN",
     "Full/no-topic/no-pattern/no-redundancy with Run82 parity guard"],
]
REVIEW_STATUS = pd.DataFrame(status_rows, columns=["robustness_task","status","evidence"])
display(REVIEW_STATUS)
REVIEW_STATUS.to_csv(OUT / "90_robustness_completion_status.csv", index=False)

In [ ]:
#@title 26. Export one manuscript/robustness Excel workbook
xlsx = OUT / "Paper2A_IJIES_Robustness_Tables_FINAL.xlsx"

with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
    REVIEW_STATUS.to_excel(xw, "Robustness_Status", index=False)
    metric_reconciliation.to_excel(xw, "Metric_Reconciliation", index=False)
    R3B_CMP.to_excel(xw, "Round3B_Definitive", index=False)
    R3C_CMP.to_excel(xw, "Round3C_Definitive", index=False)

    optional = [
        ("13_primera_fair_budget_comparison.csv","PRIMERA"),
        ("22_rebel_run82_vs_leadw_summary.csv","REBEL"),
        ("32_webnlg_true_regrouping_summary.csv","WebNLG_Regrouping"),
        ("43_ablation_means.csv","Ablation_Means"),
        ("44_ablation_paired_comparisons.csv","Ablation_Paired"),
        ("45_ablation_strict5pct_sensitivity.csv","Ablation_Strict5"),
    ]
    for fname, sheet in optional:
        p = OUT / fname
        if p.exists():
            pd.read_csv(p).to_excel(xw, sheet, index=False)

print("Saved:", xlsx)

In [ ]:
#@title 27. Generate manuscript-ready result sentences automatically
sentences = []

if (OUT / "13_primera_fair_budget_comparison.csv").exists():
    p = pd.read_csv(OUT / "13_primera_fair_budget_comparison.csv")
    sentences.append(
        "Modern baseline: report PRIMERA as a contemporary fair-budget comparator; "
        "describe metric-by-metric effects rather than claiming overall superiority."
    )

if (OUT / "22_rebel_run82_vs_leadw_summary.csv").exists():
    r = pd.read_csv(OUT / "22_rebel_run82_vs_leadw_summary.csv")
    sentences.append(
        "Alternative extractor: interpret whether component-level and complete-triple "
        "effects retain the same qualitative ordering under REBEL."
    )

if (OUT / "32_webnlg_true_regrouping_summary.csv").exists():
    w = pd.read_csv(OUT / "32_webnlg_true_regrouping_summary.csv")
    sentences.append(
        "WebNLG regrouping: report the fraction of alternative bundle constructions "
        "that preserve the direction and statistical detectability of each effect."
    )

if (OUT / "44_ablation_paired_comparisons.csv").exists():
    a = pd.read_csv(OUT / "44_ablation_paired_comparisons.csv")
    sentences.append(
        "Ablation: attribute preservation gains only to components supported by the paired "
        "ablation evidence; do not generalize the hierarchy beyond the tested selector family."
    )

text = "\n".join(f"- {s}" for s in sentences)
(OUT / "91_manuscript_interpretation_notes.txt").write_text(text, encoding="utf-8")
print(text)

In [ ]:
#@title 28. Package all outputs and download
manifest = {
    "historical_run82_seed": 42,
    "robustness_analysis_seed": 2026,
    "run82": {
        "alpha_topic":0.30,
        "beta_pattern":0.70,
        "delta_redundancy":1.00,
        "summary_sentence_budget":15,
        "n_topics":25,
    },
    "modern_baseline_n": MODERN_BASELINE_N,
    "rebel_n": REBEL_N,
    "ablation_n": ABLATION_N,
    "webnlg_regroupings": WEBNLG_REGROUPINGS,
}
(OUT / "RUN_MANIFEST.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

archive = shutil.make_archive(
    "/content/Paper2A_IJIES_Robustness_Analysis_FINAL_OUTPUTS",
    "zip",
    root_dir=OUT,
)
print("Created:", archive)

from google.colab import files
files.download(archive)

# Completion gate

For resubmission, all four robustness experiments should read **PASS** in `90_robustness_completion_status.csv`.

Then update:

1. **Abstract** — modest component gains; no unsupported structural-superiority claim.
2. **Methods** — modern comparator, alternative extractor, regrouping protocol, and ablation protocol.
3. **Results** — PRIMERA, REBEL, regrouping, and ablation tables.
4. **Discussion** — separate statistical detectability from practical magnitude.
5. **Threats to validity** — extractor dependence, benchmark/domain dependence, and grouping assumptions.
6. **Conclusion** — describe the preservation hierarchy only as an empirical pattern supported under the evaluated methods.
7. **Response letter** — replace every `[INSERT ...]` with the actual exported values.
8. **IJIES manuscript formatting** — mark all revision text in red for the second review.